# LangChain Memory 시리즈 2/7: ConversationBufferWindowMemory

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- 슬라이딩 윈도우(Sliding Window) 방식이 왜 필요한지 설명할 수 있어요
- `k` 파라미터로 유지할 대화 턴 수를 조절할 수 있어요
- `ConversationBufferMemory`와 비교해 장단점을 설명할 수 있어요
- 서비스 요건에 맞는 `k` 값을 선택하는 기준을 세울 수 있어요

## 사전 지식

| 개념 | 설명 |
|------|------|
| ConversationBufferMemory | 전체 대화를 모두 저장하는 기본 메모리 (1번 노트북) |
| 슬라이딩 윈도우 (Sliding Window) | 고정 크기의 창이 순서대로 이동하며 데이터를 처리하는 방식 |
| 토큰 예산 (Token Budget) | LLM API 호출 1회에 사용 가능한 최대 토큰 수 |

## 왜 윈도우가 필요할까요?

`ConversationBufferMemory`는 대화가 쌓일수록 프롬프트에 포함되는 이력이 길어져요. 결국 **토큰 제한**에 걸리거나, API 비용이 급증하는 문제가 생겨요.

`ConversationBufferWindowMemory`는 **슬라이딩 윈도우(Sliding Window)** 방식으로 최근 `k`개 대화만 유지해요. 오래된 대화는 자동으로 버려지기 때문에 토큰 사용량이 일정하게 유지돼요.

```mermaid
flowchart LR
    subgraph Before["5턴 대화 후 (k=2)"]
        T1[1턴: 제주도 추천] -.삭제.- X1[ ]
        T2[2턴: 3박4일 계획] -.삭제.- X2[ ]
        T3[3턴: 렌터카 질문] -.삭제.- X3[ ]
        T4[4턴: 숙소 추천]
        T5[5턴: 맛집 질문]
    end

    T4 & T5 --> WIN[윈도우 내 유지]
    WIN --> LLM[LLM 프롬프트]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    class T4,T5 input
    class WIN process
    class LLM output
    class T1,T2,T3 error
```

> **참고**: 이 노트북은 레거시 메모리 클래스를 사용해요. LangChain 1.0.x에서는 메시지 트리밍 미들웨어 패턴이 권장돼요. 최신 방식은 12번 노트북을 참고하세요.

In [ ]:
# ---------------------------------------------------
# 레거시 메모리 모듈 임포트
# ---------------------------------------------------
# ⚠️ LangChain 1.0.x에서는 더 이상 `langchain.memory` 패키지를 사용하지 않습니다.
#    구식 메모리 클래스를 시연하기 위해 `langchain.memory`에서 불러옵니다.
from langchain.memory import ConversationBufferWindowMemory

## 1. 기본 사용법 - 최근 K개 대화만 유지

`k` 파라미터로 유지할 대화 턴의 개수를 지정합니다.  
예를 들어 `k=2`로 설정하면 최근 2개의 대화 턴만 저장됩니다.


### 💡 슬라이딩 윈도우 기본 흐름 (k=2)

```mermaid
flowchart TD
    U1[사용자 입력] --> SAVE1[save_context()]
    A1[AI 응답] --> SAVE1
    SAVE1 --> HIST1[history]
    HIST1 --> CHECK1{턴 수 > k?}
    CHECK1 -- 예 --> DROP1[가장 오래된 턴 삭제]
    DROP1 --> HIST1
    CHECK1 -- 아니오 --> KEEP1[최근 k개 유지]
    KEEP1 --> LOAD1["load_memory_variables({})"]
    LOAD1 --> OUTPUT1[최근 2개 턴만 반환]
```



In [ ]:
# ============================================================
# TODO: k=2로 슬라이딩 윈도우 메모리를 생성하고 5개 대화를 저장하세요
# 힌트: ConversationBufferWindowMemory(k=2, return_messages=True)
# 예상 결과: 5개 대화 저장 후 최근 2개 대화 턴(메시지 4개)만 남음
# ============================================================

# 1단계: 최근 2개의 대화만 유지하는 메모리 생성
# - k=2: 최근 2개 대화 턴(사용자+AI 쌍 2개 = 메시지 4개) 유지
# - k를 넘어서면 가장 오래된 턴부터 자동 삭제


# 2단계: 여러 대화를 순차적으로 저장
conversations = [
    {
        "human": "안녕하세요! 요리 레시피를 추천해주세요.",
        "ai": "안녕하세요! 어떤 요리를 원하시나요? 한식, 양식, 중식, 일식 중에서 선택해주세요."
    },
    {
        "human": "한식이 좋아요. 간단하게 만들 수 있는 반찬을 추천해주세요.",
        "ai": "한식 반찬이라면 시금치나물이나 콩나물무침이 간단하고 맛있어요. 시금치나물은 데친 시금치에 다진 마늘, 참기름, 깨소금을 넣어 무치면 됩니다."
    },
    {
        "human": "시금치나물 레시피를 자세히 알려주세요.",
        "ai": "시금치나물 만드는 법: 1) 시금치를 끓는 물에 데친 후 찬물에 헹구고 물기를 꽉 짜세요. 2) 다진 마늘 1작은술, 참기름 1큰술, 깨소금 약간을 넣고 무치세요. 3) 소금으로 간을 맞추면 완성입니다!"
    },
    {
        "human": "콩나물무침도 알려주세요.",
        "ai": "콩나물무침 만드는 법: 1) 콩나물을 끓는 물에 데치고 찬물에 헹구세요. 2) 다진 마늘, 대파, 고춧가루, 참기름, 깨소금을 넣고 무치세요. 3) 소금과 식초로 간을 맞추면 됩니다."
    },
    {
        "human": "두 반찬 모두 만들 시간이 얼마나 걸리나요?",
        "ai": "시금치나물은 약 10분, 콩나물무침은 약 15분 정도 걸립니다. 둘 다 간단하고 빠르게 만들 수 있어요!"
    }
]

# 3단계: 모든 대화 저장 (k=2를 넘으면 오래된 턴은 자동 삭제됨)


### 저장된 대화 확인

`k=2`로 설정했으므로, **최근 2개의 대화 턴만** 저장되어 있는지 확인해봅시다.


In [ ]:
# ---------------------------------------------------
# 슬라이딩 윈도우 결과 확인 (k=2)
# ---------------------------------------------------
# 5개 대화를 저장했지만 k=2이므로 최근 2턴(메시지 4개)만 유지됨


💡 **확인 결과**: 총 5개의 대화를 저장했지만, `k=2`로 설정했기 때문에 **최근 2개의 대화 턴(4개의 메시지)**만 저장되어 있습니다.  
오래된 대화는 자동으로 삭제되어 메모리에서 제거되었습니다.


## 2. k 값에 따른 차이 비교

다양한 `k` 값으로 설정했을 때 어떤 대화가 저장되는지 비교해봅시다.


### 💡 `k` 값 실험 루프 구조

```mermaid
flowchart LR
    subgraph LoopTest[각 k 값 테스트]
        KLIST["[k ∈ {1,2,3}"]] --> INIT[메모리 생성]
        INIT --> SAVESEQ[4개의 대화 save_context]
        SAVESEQ --> TRIM[윈도우 로직 적용]
        TRIM --> LOADSEQ["load_memory_variables({})"]
        LOADSEQ --> PRINTSEQ[결과 출력]
    end
```



In [ ]:
# ============================================================
# TODO: k=1, 2, 3 세 가지 값으로 메모리를 생성해 동일 대화를 저장하고 결과를 비교하세요
# 힌트: for k_value in [1, 2, 3]: 루프 → 메모리 생성 → 대화 저장 → 메시지 수 출력
# 예상 결과: k=1이면 메시지 2개, k=2이면 4개, k=3이면 6개(또는 전체) 남음
# ============================================================

# 테스트용 대화 데이터 (4턴)
test_conversations = [
    {"human": "첫 번째 질문입니다.", "ai": "첫 번째 답변입니다."},
    {"human": "두 번째 질문입니다.", "ai": "두 번째 답변입니다."},
    {"human": "세 번째 질문입니다.", "ai": "세 번째 답변입니다."},
    {"human": "네 번째 질문입니다.", "ai": "네 번째 답변입니다."},
]

# k=1, k=2, k=3으로 각각 테스트
# - k=1: 최근 1개 대화 턴(메시지 2개) 유지
# - k=2: 최근 2개 대화 턴(메시지 4개) 유지
# - k=3: 최근 3개 대화 턴(메시지 6개) 유지


### 💡 두 메모리 타입 비교 흐름

```mermaid
flowchart LR
    subgraph InputSeq[동일 대화 입력 루프]
        START[[comparison_conversations]] --> SAVE_BUF[save_context → buffer_memory]
        START --> SAVE_WIN[save_context → window_memory]
    end

    SAVE_BUF --> BUF_STORE[전체 history 유지]
    SAVE_WIN --> WIN_STORE[최근 2턴만 유지]

    BUF_STORE --> LOAD_BUF["load_memory_variables({})"]
    WIN_STORE --> LOAD_WIN["load_memory_variables({})"]

    LOAD_BUF --> REPORT_BUF[전체 메시지 수 출력]
    LOAD_WIN --> REPORT_WIN[최근 메시지 수 출력]
```



## 3. ConversationBufferMemory와 비교

`ConversationBufferMemory`와 `ConversationBufferWindowMemory`의 차이를 비교해봅시다.


In [ ]:
# ---------------------------------------------------
# ConversationBufferMemory vs ConversationBufferWindowMemory 비교
# ---------------------------------------------------
# 동일한 4개 대화를 두 메모리에 저장하여 보관 개수 차이를 확인

from langchain.memory import ConversationBufferMemory

# 비교용 대화 데이터
comparison_conversations = [
    {"human": "오늘 날씨가 어때요?", "ai": "오늘은 맑고 따뜻한 날씨입니다."},
    {"human": "내일은 어떤가요?", "ai": "내일은 비가 올 예정입니다."},
    {"human": "주말 날씨는요?", "ai": "주말에는 구름이 많을 것 같습니다."},
    {"human": "다음 주는요?", "ai": "다음 주는 대체로 맑을 예정입니다."},
]

# ConversationBufferMemory: 모든 대화 저장 (제한 없음)


# ConversationBufferWindowMemory: 최근 2개만 저장


# 두 메모리에 동일한 대화 저장


# 결과 비교


## 4. 실전 예제: 고객 상담 챗봇

긴 대화가 발생할 수 있는 고객 상담 시나리오에서 `ConversationBufferWindowMemory`를 사용하는 예제입니다.


### 💡 고객 상담 시나리오 메모리 흐름 (k=3)

```mermaid
flowchart TD
    subgraph Session[고객 상담 루프]
        QH[고객 문의] --> SAVE_C[save_context()]
        SAVE_C --> QA[상담사 응답 저장]
        QA --> HIST_C[customer_memory.history]
        HIST_C --> CHECK_C{턴 수 > 3?}
        CHECK_C -- 예 --> DROP_C[가장 오래된 상담 기록 제거]
        DROP_C --> HIST_C
        CHECK_C -- 아니오 --> CONTINUE_C[최근 3턴 유지]
    end

    CONTINUE_C --> LOAD_C["load_memory_variables({})"]
    LOAD_C --> VIEW_C[최근 3개 대화 출력]
```



In [ ]:
# ============================================================
# TODO: k=3으로 고객 상담 메모리를 생성하고 6개 대화를 저장하세요
# 힌트: ConversationBufferWindowMemory(k=3, return_messages=True)
# 예상 결과: 6개 대화 저장 후 최근 3개 대화 턴(메시지 6개)만 남음
# ============================================================

# 1단계: 고객 상담 메모리 생성 (최근 3개 대화만 유지)
# - k=3: 최근 3턴을 유지하므로 토큰 예산을 고정하면서 충분한 맥락 제공


# 2단계: 고객 상담 대화 시뮬레이션 데이터 (6턴)
customer_conversations = [
    {
        "human": "안녕하세요. 제품 배송 문의드려요.",
        "ai": "안녕하세요! 배송 관련 문의를 도와드리겠습니다. 주문번호를 알려주시면 확인해드릴게요."
    },
    {
        "human": "주문번호는 ORD-12345입니다.",
        "ai": "주문번호 ORD-12345로 확인했습니다. 현재 배송 중이며 내일 도착 예정입니다."
    },
    {
        "human": "배송 주소를 변경하고 싶어요.",
        "ai": "배송 주소 변경이 가능합니다. 새로운 주소를 알려주시면 변경해드리겠습니다."
    },
    {
        "human": "서울시 강남구 테헤란로 123으로 변경해주세요.",
        "ai": "주소를 서울시 강남구 테헤란로 123으로 변경했습니다. 배송지 변경으로 인해 배송이 하루 지연될 수 있습니다."
    },
    {
        "human": "변경된 주소로 언제 도착하나요?",
        "ai": "변경된 주소로는 모레 도착 예정입니다. 배송 추적은 주문번호로 확인하실 수 있습니다."
    },
    {
        "human": "배송비는 추가로 발생하나요?",
        "ai": "주소 변경으로 인한 추가 배송비는 없습니다. 기존 배송비만 적용됩니다."
    }
]

# 3단계: 모든 대화 저장 (k=3 이상이면 오래된 턴 자동 삭제)


## 5. k 값 선택 가이드

적절한 `k` 값을 선택하는 방법을 알아봅시다.


In [ ]:
# ---------------------------------------------------
# k 값 선택 가이드 출력
# ---------------------------------------------------
# k 값은 서비스 특성과 토큰 예산에 따라 설정
# 1턴 = 사용자+AI 메시지 쌍 1개
print("=" * 60)
print("📋 k 값 선택 가이드")
print("=" * 60)

guidelines = [
    {
        "k": 1,
        "설명": "최근 1개 대화만 유지",
        "장점": "메모리 사용량 최소화, 토큰 절약",
        "단점": "컨텍스트가 매우 제한적",
        "적용": "간단한 질의응답, 컨텍스트가 거의 필요 없는 경우"
    },
    {
        "k": 2,
        "설명": "최근 2개 대화 유지",
        "장점": "메모리 효율적이면서 최소한의 컨텍스트 유지",
        "단점": "여전히 제한적인 컨텍스트",
        "적용": "짧은 대화, 간단한 상담"
    },
    {
        "k": 3,
        "설명": "최근 3개 대화 유지",
        "장점": "적절한 컨텍스트와 메모리 효율성의 균형",
        "단점": "복잡한 대화에서는 부족할 수 있음",
        "적용": "일반적인 고객 상담, 중간 길이 대화"
    },
    {
        "k": 5,
        "설명": "최근 5개 대화 유지",
        "장점": "충분한 컨텍스트 제공",
        "단점": "메모리 사용량 증가",
        "적용": "복잡한 상담, 긴 대화"
    }
]

for guide in guidelines:
    print(f"\n🔹 k={guide['k']}: {guide['설명']}")
    print(f"   ✅ 장점: {guide['장점']}")
    print(f"   ⚠️  단점: {guide['단점']}")
    print(f"   💡 적용: {guide['적용']}")

print("\n" + "=" * 60)
print("💡 권장사항")
print("=" * 60)
print("• 짧은 대화: k=1~2")
print("• 일반적인 대화: k=3~5")
print("• 복잡한 대화: k=5~10")
print("• 매우 긴 대화: ConversationSummaryMemory 또는 VectorStoreRetrieverMemory 고려")

## 핵심 정리

```python
# ConversationBufferWindowMemory 기본 사용법
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=2, return_messages=True)

# 대화 저장
memory.save_context(
    inputs={"human": "사용자 메시지"},
    outputs={"ai": "AI 응답"}
)

# 대화 로드 (최근 k개만 반환)
history = memory.load_memory_variables({})["history"]
```

### 주요 특징
- ✅ **슬라이딩 윈도우**: 최근 K개의 대화만 유지
- ✅ **메모리 효율성**: 오래된 대화 자동 삭제로 메모리 사용량 제어
- ✅ **토큰 제한 대응**: 긴 대화에서도 토큰 사용량 일정하게 유지
- ✅ **간단한 설정**: `k` 파라미터로 유지할 대화 개수 조절

### ConversationBufferMemory와의 차이

| 특징 | ConversationBufferMemory | ConversationBufferWindowMemory |
|------|-------------------------|-------------------------------|
| 저장 방식 | 전체 대화 저장 | 최근 K개만 저장 |
| 메모리 사용 | 증가함 (대화가 길어질수록) | 일정함 (K개로 제한) |
| 토큰 사용 | 증가함 | 일정함 |
| 컨텍스트 | 전체 컨텍스트 유지 | 최근 컨텍스트만 유지 |
| 적합한 경우 | 짧은 대화 | 긴 대화, 토큰 제한 관리 |

### 주의사항
- ⚠️ **k 값 선택**: 너무 작으면 컨텍스트 부족, 너무 크면 메모리 낭비
- ⚠️ **오래된 정보 손실**: K개를 넘어가는 오래된 대화는 완전히 삭제됨
- ⚠️ **중요 정보 보존**: 중요한 정보는 별도로 저장하거나 다른 메모리 타입 고려